In [ ]:
!pip install transformers peft torch fastapi uvicorn \
            pyngrok huggingface_hub bitsandbytes \
            accelerate sentencepiece -q
print("Installation Done!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 16.2 MB/s eta 0:00:00
Installation Done!


In [ ]:
from huggingface_hub import login
login()

Loading model


In [ ]:
import torch
import threading
import uuid
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional
import uvicorn

BASE_MODEL = "meta-llama/Llama-3.1-8B-Instruct"
ADAPTER_MODEL = "mamoon94/greenenergy-ev-assistant"

SYSTEM_PROMPT = """You are GreenEnergy, a Smart EV Assistant with expert
knowledge of electric vehicle technology, battery systems, charging
infrastructure, grid optimization, vehicle specifications, performance
metrics, costs, and sustainability. Provide accurate, detailed answers."""

print("Loading model... (takes 5-10 mins first time)")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_MODEL)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
model = PeftModel.from_pretrained(model, ADAPTER_MODEL)
model.eval()
print("✅ Model loaded!")

app = FastAPI()
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

sessions = {}


class ChatRequest(BaseModel):
    session_id: Optional[str] = None
    question: str
    max_tokens: Optional[int] = 512
    temperature: Optional[float] = 0.7


@app.get("/")
def root():
    return {"status": "online", "agent": "GreenEnergy"}


@app.post("/chat")
def chat(request: ChatRequest):
    session_id = request.session_id or str(uuid.uuid4())
    if session_id not in sessions:
        sessions[session_id] = []
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    messages.extend(sessions[session_id][-10:])
    messages.append({"role": "user", "content": request.question})
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=request.max_tokens,
            temperature=request.temperature,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    sessions[session_id].append({"role": "user", "content": request.question})
    sessions[session_id].append({"role": "assistant", "content": response})
    return {"session_id": session_id, "answer": response, "question": request.question}


@app.delete("/chat/{session_id}")
def clear(session_id: str):
    if session_id in sessions:
        del sessions[session_id]
    return {"message": "cleared"}


def run():
    uvicorn.run(app, host="0.0.0.0", port=8000)


thread = threading.Thread(target=run, daemon=True)
thread.start()
print("API running!")

Loading model... (takes 5-10 mins first time)


tokenizer_config.json:   0%|          | 0.00/353 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/4.61k [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 83.9MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

✅ Model loaded!
API running!


In [ ]:
from pyngrok import ngrok

# Paste your token here
ngrok.set_auth_token("enter token here")

public_url = ngrok.connect(8000)
print(f"✅ Your ngrok URL: {public_url}")
print("Copy this URL!")

✅ Your ngrok URL: NgrokTunnel: "https://bless-brought-slip.ngrok-free.dev" -> "http://localhost:8000"
Copy this URL!


In [ ]:
from pyngrok import ngrok

# Kill existing tunnel
ngrok.kill()

# Reconnect with proper config
public_url = ngrok.connect(8000, bind_tls=True)
print(f"✅ New URL: {public_url}")

✅ New URL: NgrokTunnel: "https://bless-brought-slip.ngrok-free.dev" -> "http://localhost:8000"


In [ ]:
import requests
response = requests.get("https://bless-brought-slip.ngrok-free.dev/",
                        headers={"ngrok-skip-browser-warning": "true"})
print(response.json())

INFO:     35.187.253.216:0 - "GET / HTTP/1.1" 200 OK
{'status': 'online', 'agent': 'GreenEnergy'}
